# ViFinQA CCL Phase 5 bounded component selection

This private P100 experiment runs the bounded hash-bound source-component packet set declared by its immutable job manifest. The model may only select existing literal component IDs or use a closed-world abstention. It cannot emit a free table-semantic label. Every output remains proposal-only, non-training and non-certified.

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tarfile

WORKING = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
SOURCE_DIR = WORKING / 'ai_guru_ccl_phase5_component_selection_source'
RAW_DIR = WORKING / 'ccl_phase5_component_selection_smoke_raw'
VALIDATED_DIR = WORKING / 'ccl_phase5_component_selection_smoke_validated'

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for this CCL smoke. Enable a Kaggle GPU accelerator.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
if GPU_VRAM_GIB < 14:
    raise RuntimeError(f'The declared 4-bit route requires at least 14 GiB VRAM; observed {GPU_VRAM_GIB} GiB.')
print({'gpu': GPU_NAME, 'vram_gib': GPU_VRAM_GIB, 'python': sys.version.split()[0]})


## Required private inputs

Attach exactly two private datasets: the minimal source bundle and one hash-bound component-selection job. Internet is used only to download the model declared by that job manifest.

In [ ]:
SOURCE_ARCHIVE_NAME = 'ai_guru_ccl_phase3_source_v1.bundle'
SOURCE_MANIFEST_NAME = 'ai_guru_ccl_phase3_source_v1.manifest.json'

def exactly_one_input(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one Kaggle input named {name!r}; found {matches}')
    return matches[0]

def exactly_one_input_suffix(suffix: str) -> Path:
    matches = sorted(path for path in INPUT_ROOT.rglob('*') if path.is_file() and path.name.endswith(suffix))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one Kaggle input ending {suffix!r}; found {matches}')
    return matches[0]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_ARCHIVE = exactly_one_input(SOURCE_ARCHIVE_NAME)
SOURCE_MANIFEST = exactly_one_input(SOURCE_MANIFEST_NAME)
JOB_MANIFEST = exactly_one_input_suffix('_job_manifest.json')
job_manifest = json.loads(JOB_MANIFEST.read_text(encoding='utf-8'))
JOB_CONTRACTS = {'vifinqa_ccl_phase5_component_selection_smoke_v1': ('prepared_component_selection_smoke_not_executed', 5, 5), 'vifinqa_ccl_phase5_component_selection_pilot_v1': ('prepared_component_selection_pilot_not_executed', 1, 32)}
job_contract = JOB_CONTRACTS.get(job_manifest.get('protocol'))
if (job_contract is None or job_manifest.get('run_status') != job_contract[0] or job_manifest.get('model_execution_allowed') is not True or job_manifest.get('training_eligible') is not False or job_manifest.get('certification_allowed') is not False):
    raise RuntimeError('Job is not a proposal-only prepared component-selection job.')
route = job_manifest.get('route') or {}
ROUTE_ID = route.get('route_id')
EXPECTED_PACKET_COUNT = len(route.get('packet_ids') or [])
if not isinstance(ROUTE_ID, str) or not job_contract[1] <= EXPECTED_PACKET_COUNT <= job_contract[2]:
    raise RuntimeError('Job route has an unsupported packet count.')
job_outputs = job_manifest.get('outputs') or {}
job_input_names = [name for name in job_outputs if name.endswith('.jsonl') or name.endswith('_packet_manifest.json')]
if (len(job_input_names) != 3 or sum(name.endswith('_requests_v1.jsonl') for name in job_input_names) != 1 or sum(name.endswith('_packets_v1.jsonl') for name in job_input_names) != 1 or sum(name.endswith('_packet_manifest.json') for name in job_input_names) != 1):
    raise RuntimeError('Job does not bind exactly one requests, packets and packet manifest input.')
REQUESTS_NAME = next(name for name in job_input_names if name.endswith('_requests_v1.jsonl'))
PACKETS_NAME = next(name for name in job_input_names if name.endswith('_packets_v1.jsonl'))
PACKET_MANIFEST_NAME = next(name for name in job_input_names if name.endswith('_packet_manifest.json'))
REQUESTS = exactly_one_input(REQUESTS_NAME)
PACKETS = exactly_one_input(PACKETS_NAME)
PACKET_MANIFEST = exactly_one_input(PACKET_MANIFEST_NAME)
source_manifest = json.loads(SOURCE_MANIFEST.read_text(encoding='utf-8'))
if source_manifest.get('protocol') != 'kaggle_ccl_phase3_source_bundle_v1':
    raise RuntimeError('Unexpected source-bundle protocol.')
archive_contract = (source_manifest.get('outputs') or {}).get('archive') or {}
if archive_contract.get('sha256') != sha256_file(SOURCE_ARCHIVE):
    raise RuntimeError('Source bundle archive does not match its manifest SHA-256.')
if SOURCE_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite source directory: {SOURCE_DIR}')
identity = source_manifest.get('source_bundle') or {}
files = identity.get('files') or []
SOURCE_DIR.mkdir(parents=True)
with tarfile.open(SOURCE_ARCHIVE, mode='r:*') as archive:
    expected_names = [entry['path'] for entry in files] + ['SOURCE_BUNDLE.json']
    members = archive.getmembers()
    if [member.name for member in members] != expected_names or not all(member.isfile() for member in members):
        raise RuntimeError('Source archive members do not exactly match the source identity.')
    for entry in files:
        name, expected_sha = entry['path'], entry['sha256']
        if Path(name).is_absolute() or '..' in Path(name).parts:
            raise RuntimeError(f'Unsafe source member path: {name}')
        handle = archive.extractfile(name)
        if handle is None or hashlib.sha256(handle.read()).hexdigest() != expected_sha:
            raise RuntimeError(f'Source member checksum mismatch: {name}')
        handle = archive.extractfile(name)
        destination = SOURCE_DIR / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(handle.read())
for path in (REQUESTS, PACKETS, PACKET_MANIFEST):
    if ((job_manifest.get('outputs') or {}).get(path.name) or {}).get('sha256') != sha256_file(path):
        raise RuntimeError(f'Job output checksum mismatch: {path.name}')
packet_manifest = json.loads(PACKET_MANIFEST.read_text(encoding='utf-8'))
if (packet_manifest.get('protocol') != 'vifinqa_ccl_phase5_component_selection_v1'
        or packet_manifest.get('selected_packet_ids') != (job_manifest.get('route') or {}).get('packet_ids')
        or ((packet_manifest.get('outputs') or {}).get(PACKETS.name) or {}).get('sha256') != sha256_file(PACKETS)):
    raise RuntimeError('Selected packet manifest does not bind the exact component-selection packet set.')
print({'source_tree_sha256': identity.get('source_tree_sha256'), 'job_manifest_sha256': sha256_file(JOB_MANIFEST), 'request_count': sum(1 for line in REQUESTS.open(encoding='utf-8') if line.strip())})


## P100-compatible 4-bit runtime

The preflight requires an SM60-capable CUDA wheel before checkpoint download. Dependency installs use `--no-deps` after the pinned PyTorch pair.

In [ ]:
GPU_CAPABILITY = tuple(int(value) for value in torch.cuda.get_device_capability(0))
if GPU_CAPABILITY < (6, 0):
    raise RuntimeError(f'NF4 4-bit requires compute capability >= 6.0; observed {GPU_CAPABILITY}.')
RUNTIME_LOCK = {'torch': 'torch==2.6.0', 'torchvision': 'torchvision==0.21.0', 'torch_index_url': 'https://download.pytorch.org/whl/cu118', 'tokenizers': 'tokenizers==0.21.4', 'huggingface_hub': 'huggingface-hub==0.30.2', 'safetensors': 'safetensors==0.5.3', 'transformers': 'transformers==4.51.3', 'accelerate': 'accelerate==1.7.0', 'bitsandbytes': 'bitsandbytes==0.45.5', 'sentencepiece': 'sentencepiece==0.2.0'}
if GPU_CAPABILITY < (7, 0):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall', '--index-url', RUNTIME_LOCK['torch_index_url'], RUNTIME_LOCK['torch']], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall', '--no-deps', '--index-url', RUNTIME_LOCK['torch_index_url'], RUNTIME_LOCK['torchvision']], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall', '--no-deps', RUNTIME_LOCK['tokenizers'], RUNTIME_LOCK['huggingface_hub'], RUNTIME_LOCK['safetensors'], RUNTIME_LOCK['transformers'], RUNTIME_LOCK['accelerate'], RUNTIME_LOCK['bitsandbytes'], RUNTIME_LOCK['sentencepiece']], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE_DIR), '--no-deps'], check=True)
RUNTIME_PROBE = """
import json, torch, torchvision, bitsandbytes
if not torch.cuda.is_available(): raise RuntimeError('CUDA disappeared after runtime resolution.')
capability = tuple(int(value) for value in torch.cuda.get_device_capability(0))
arch_list = list(torch.cuda.get_arch_list()); expected_arch = f'sm_{capability[0]}{capability[1]}'
if capability < (6, 0) or expected_arch not in arch_list: raise RuntimeError(f'Installed PyTorch cannot run NF4 on {expected_arch}: {arch_list}')
if capability < (7, 0) and not torch.__version__.startswith('2.6.0+cu118'): raise RuntimeError(f'P100 PyTorch runtime was overwritten: {torch.__version__}')
if capability < (7, 0) and not torchvision.__version__.startswith('0.21.0+cu118'): raise RuntimeError(f'P100 torchvision runtime is mismatched: {torchvision.__version__}')
torch.empty((1,), device='cuda').zero_()
print(json.dumps({'cuda_available': True, 'torch_version': torch.__version__, 'torchvision_version': torchvision.__version__, 'torch_cuda_version': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0), 'gpu_compute_capability': list(capability), 'torch_arch_list': arch_list, 'bitsandbytes_version': bitsandbytes.__version__}))
"""
probe = subprocess.run([sys.executable, '-c', RUNTIME_PROBE], capture_output=True, text=True)
if probe.returncode:
    print(probe.stdout); print(probe.stderr, file=sys.stderr)
    raise RuntimeError(f'CCL model runtime preflight failed with exit code {probe.returncode}.')
MODEL_RUNTIME = json.loads(probe.stdout.strip().splitlines()[-1])
print({'runtime_lock': RUNTIME_LOCK, 'model_runtime': MODEL_RUNTIME})


## Execute and validate

Raw model text is persisted first. The closed-world validator then accepts only IDs from the immutable bounded packet menu. Valid output is still component-selection-only, not a semantic certificate.

In [ ]:
subprocess.run([sys.executable, str(SOURCE_DIR / 'scripts/run_ccl_phase5_component_selection_model.py'), '--job-manifest', str(JOB_MANIFEST), '--requests', str(REQUESTS), '--output-dir', str(RAW_DIR), '--progress-every', '1'], check=True)
subprocess.run([sys.executable, str(SOURCE_DIR / 'scripts/validate_ccl_phase5_component_selection_responses.py'), '--packets', str(PACKETS), '--packet-manifest', str(PACKET_MANIFEST), '--raw-responses', str(RAW_DIR / 'component_selection_raw_responses_v1.jsonl'), '--route-id', ROUTE_ID, '--output-dir', str(VALIDATED_DIR)], check=True)
execution_report = json.loads((RAW_DIR / 'component_selection_model_execution_report.json').read_text(encoding='utf-8'))
validation_manifest = json.loads((VALIDATED_DIR / 'phase5_component_selection_validation_manifest.json').read_text(encoding='utf-8'))
validation_report = json.loads((VALIDATED_DIR / 'phase5_component_selection_validation_report.json').read_text(encoding='utf-8'))
if execution_report.get('response_count') != EXPECTED_PACKET_COUNT or sum((validation_report.get('status_counts') or {}).values()) != EXPECTED_PACKET_COUNT:
    raise RuntimeError('Execution or validation did not cover the exact declared packet set.')
if validation_manifest.get('training_eligible') is not False or validation_manifest.get('certification_allowed') is not False:
    raise RuntimeError('Validator receipt violates the non-promotable contract.')
runtime = execution_report.get('runtime') or {}
if runtime.get('gpu_name') != MODEL_RUNTIME['gpu_name'] or runtime.get('torch_version') != MODEL_RUNTIME['torch_version']:
    raise RuntimeError('Execution runtime differs from the runtime preflight.')
receipt = {'schema_version': 1, 'protocol': 'vifinqa_ccl_phase5_component_selection_kaggle_kernel_v1', 'gpu': {'name': GPU_NAME, 'vram_gib': GPU_VRAM_GIB}, 'model_runtime': MODEL_RUNTIME, 'source_tree_sha256': identity['source_tree_sha256'], 'job_manifest_sha256': sha256_file(JOB_MANIFEST), 'execution_report_sha256': sha256_file(RAW_DIR / 'component_selection_model_execution_report.json'), 'validation_manifest_sha256': sha256_file(VALIDATED_DIR / 'phase5_component_selection_validation_manifest.json'), 'validation_status_counts': validation_report.get('status_counts'), 'training_eligible': False, 'certification_allowed': False}
(WORKING / 'ccl_phase5_component_selection_kaggle_receipt_v1.json').write_text(json.dumps(receipt, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(receipt, ensure_ascii=False, indent=2))
print('Download RAW_DIR, VALIDATED_DIR and ccl_phase5_component_selection_kaggle_receipt_v1.json. Do not promote or fine-tune from this bounded run.')
